# 01 — Run canonical calibration

Runs the two fixed pricing specifications on the complete snapshot dataset:

- **WLS:** `lambda_TS = 0` → `excel files/canonical/calibration_wls.xlsx`
- **PWLS:** `lambda_TS = 1` → `excel files/canonical/calibration_pwls.xlsx`

Both use the deterministic **70/30 train/test** split from the core pipeline. There is no third split. By default `START_DT` and `END_DT` are `None`, so the runner uses every snapshot currently present in `data/`. Set explicit dates only for a deliberately bounded robustness run.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

NOTEBOOK_ROOT = Path.cwd().resolve()
if (NOTEBOOK_ROOT / "_lib").exists():
    NOTEBOOK_DIR = NOTEBOOK_ROOT
elif (NOTEBOOK_ROOT / "notebooks" / "_lib").exists():
    NOTEBOOK_DIR = NOTEBOOK_ROOT / "notebooks"
else:
    raise FileNotFoundError(f"Could not locate notebooks/_lib from {NOTEBOOK_ROOT}")

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _lib.common import configure_notebook_display, locate_notebook_paths

PATHS = locate_notebook_paths(NOTEBOOK_DIR)
ROOT = PATHS.project_root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

configure_notebook_display(max_columns=200, width=200)
print("Project root:", ROOT)

from _lib.stage2_workflow import (
    assert_snapshot_data_ready,
    build_calibration_configs,
    inspect_snapshot_data,
    run_calibration_suite,
)


In [ ]:
START_DT = None
END_DT = None
RESUME = True

# Canonical outputs: calibration_wls.xlsx and calibration_pwls.xlsx
configs = build_calibration_configs(
    ROOT,
    start_dt=START_DT,
    end_dt=END_DT,
    resume=RESUME,
    verbose=True,
)
for key, cfg in configs.items():
    print(key.upper(), "->", cfg.output_xlsx, "lambda_TS=", cfg.l2_prev_strength)


In [ ]:
inventory = inspect_snapshot_data(PATHS.data_dir)
assert_snapshot_data_ready(inventory)
print(f"Pre-flight passed for {len(inventory)} snapshots.")


In [ ]:
calibration_outputs = run_calibration_suite(configs)
for key, cfg in configs.items():
    print(f"{key.upper()} complete: {cfg.output_xlsx}")
